# 🎵 Music to MIDI — Google Colab (Free GPU)

在 Colab 免费 T4 GPU 上运行 Music to MIDI，通过 Gradio 公开链接访问。

当前 Colab 版提供六种处理模式：
- **完整混音多乐器转写（SMART）**：直接用可选择的 YourMT3+ 官方模型模式对整首音频生成一个多轨 MIDI。
- **人声/伴奏分离后分别转写（VOCAL_SPLIT）**：先分离人声与伴奏，再分别转写并可额外合并 MIDI。
- **六声部分离 + 分别转写**：先分离 bass / drums / guitar / piano / vocals / other 六个 stem，再分别生成 stem MIDI 和合并 MIDI。
- **钢琴专用转写 (Transkun)**：使用 Transkun 钢琴专用模型直接转写纯钢琴音频。
- **钢琴专用转写 (Aria-AMT)**：使用 Aria-AMT 钢琴专用模型直接转写纯钢琴音频。

Colab 版的 SMART、VOCAL_SPLIT 与非钢琴 stem 使用可选择的 YourMT3+ 官方模型模式；钢琴专用入口使用 Transkun、Aria-AMT 或 ByteDance Pedal。桌面版如需 MIROS 后端请在本地运行。

**输出说明**：SMART 输出一个完整混音 MIDI；VOCAL_SPLIT 输出 `_accompaniment.mid` 与 `_vocal.mid`，勾选合并后额外输出 `_vocal_accompaniment_merged.mid`；六声部模式输出各 stem MIDI、分离音频和一个合并 MIDI；钢琴专用模式输出一个钢琴 MIDI。

**使用方法**：
1. 点击上方菜单 **Runtime → Change runtime type → T4 GPU**
2. 依次运行下面的单元格
3. 最后一个单元格会输出一个公开链接，打开即可使用

In [ ]:
#@title 1️⃣ 检查 GPU 可用性（详细日志）
import subprocess
import sys
from datetime import datetime

import torch


def log(message=""):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


log("Starting runtime diagnostics")
log(f"Python version: {sys.version.split()[0]}")
log(f"PyTorch version: {torch.__version__}")
log(f"Torch CUDA build: {torch.version.cuda}")
log(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_index = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_index)
    total_vram_gb = props.total_memory / 1024**3
    log(f"Detected GPU: {props.name}")
    log(f"Total VRAM: {total_vram_gb:.2f} GB")
    log(f"SM count: {props.multi_processor_count}")
    try:
        reserved = torch.cuda.memory_reserved(device_index) / 1024**3
        allocated = torch.cuda.memory_allocated(device_index) / 1024**3
        log(f"Allocated VRAM: {allocated:.3f} GB")
        log(f"Reserved VRAM: {reserved:.3f} GB")
    except Exception as exc:
        log(f"Failed to read VRAM stats: {exc}")

    log("nvidia-smi summary:")
    try:
        smi = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,memory.used,driver_version",
                "--format=csv,noheader",
            ],
            text=True,
        ).strip()
        print(smi, flush=True)
    except Exception as exc:
        log(f"Unable to run nvidia-smi: {exc}")

    log("GPU runtime check passed")
else:
    log("No GPU detected. In Colab use Runtime -> Change runtime type -> T4 GPU")
    raise RuntimeError("GPU runtime is required")


In [ ]:
#@title 2️⃣ 克隆仓库并安装依赖（详细日志）
import importlib
import os
import shlex
import subprocess
import time
from datetime import datetime
from pathlib import Path


def log(message=""):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


def run_cmd(command, cwd=None):
    log(f"CMD: {command}")
    start = time.time()
    proc = subprocess.Popen(
        command,
        cwd=cwd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    line_count = 0
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
        line_count += 1
    return_code = proc.wait()
    elapsed = time.time() - start
    log(f"command finished: exit={return_code}, lines={line_count}, elapsed={elapsed:.1f}s")
    if return_code != 0:
        raise RuntimeError(f"command failed: {command}")


repo_dir = Path("/content/music-to-midi")
repo_url = "https://github.com/mason369/music-to-midi.git"

log("准备项目工作区")
if not repo_dir.exists():
    run_cmd(f"git clone {repo_url} {repo_dir}")
else:
    log("仓库已存在，显示状态并同步远程")
    run_cmd("git -C /content/music-to-midi remote -v")
    run_cmd("git -C /content/music-to-midi rev-parse --short HEAD")
    run_cmd("git -C /content/music-to-midi fetch --all --prune")
    run_cmd("git -C /content/music-to-midi status -sb")

os.chdir(repo_dir)
log(f"工作目录: {os.getcwd()}")

log("升级 pip/setuptools/wheel")
run_cmd("python -m pip install --upgrade pip setuptools wheel")

# ── 记录 Colab 预装的 torch 版本（绝不重装） ──
log("检测 Colab 预装 torch 版本")
import torch
log(f"torch=={torch.__version__}")
log(f"CUDA available: {torch.cuda.is_available()}, CUDA version: {torch.version.cuda}")

log("安装 Python 依赖")
# 保留 Colab 预装 torch；只安装当前 Web 版需要的依赖
packages = [
    "lightning",
    "einops",
    "transformers==4.45.1",
    "mir-eval",
    "deprecated",
    "smart-open",
    "nest-asyncio",
    "scipy",
    "scikit-learn",
    "torchmetrics",
    "wandb",
    "pretty-midi>=0.2.10",
    "soxr>=0.3.7",
    "huggingface_hub",
    "mido",
    "librosa",
    "soundfile",
    "transkun>=2.0.1",
    "aria-amt @ git+https://github.com/EleutherAI/aria-amt.git",
    "piano-transcription-inference>=0.0.6,<0.1",
    "torchlibrosa>=0.1.0,<0.2",
    "gradio>=4.44.0",
    "tqdm",
    "pyyaml",
    "psutil",
    "numba",
]
quoted_packages = " ".join(shlex.quote(pkg) for pkg in packages)
run_cmd("python -m pip install " + quoted_packages)

log("安装 audio-separator 运行依赖（固定兼容 NumPy 1.26）")
audio_separator_runtime_packages = [
    "numpy==1.26.4",
    "beartype==0.18.5",
    "diffq-fixed==0.2.4",
    "julius==0.2.7",
    "ml_collections==1.1.0",
    "onnx-weekly==1.21.0.dev20260302",
    "onnx2torch-py313==1.6.0",
    "pydub==0.25.1",
    "requests>=2.32.5,<3",
    "chardet>=5,<6",
    "onnxruntime==1.23.2",
    "resampy==0.4.3",
    "rotary-embedding-torch==0.6.5",
    "samplerate==0.1.0",
    "h5py>=3.10,<4",
    "mirdata>=0.3.8,<1",
    "six==1.17.0",
]
quoted_audio_separator_runtime_packages = " ".join(shlex.quote(pkg) for pkg in audio_separator_runtime_packages)
run_cmd("python -m pip install " + quoted_audio_separator_runtime_packages)
run_cmd("python -m pip install " + shlex.quote("audio-separator==0.41.1") + " --no-deps")

log("使用 apt 安装 FFmpeg")
run_cmd("apt-get update")
run_cmd("apt-get install -y ffmpeg")

log("关键包版本")
for module_name in ["torch", "torchaudio", "torchvision", "gradio", "huggingface_hub", "lightning", "librosa"]:
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", "unknown")
        log(f"{module_name}=={version}")
    except Exception as exc:
        log(f"无法读取 {module_name} 版本: {exc}")

log("依赖安装完成")

In [ ]:
#@title 3️⃣ 下载 YourMT3+ 源码和模型权重（详细日志）
import os
import sys
import time
from datetime import datetime
from pathlib import Path

from huggingface_hub import list_repo_files, snapshot_download

os.chdir("/content/music-to-midi")


def log(message=""):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {message}", flush=True)


def format_size(num_bytes):
    size = float(num_bytes)
    for unit in ("B", "KB", "MB", "GB"):
        if size < 1024 or unit == "GB":
            if unit == "B":
                return f"{int(size)} {unit}"
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} GB"


def collect_local_file_info(base_dir, relative_paths):
    file_info = {}
    base_path = Path(base_dir)
    for relative_path in relative_paths:
        local_path = base_path / relative_path
        if local_path.exists():
            file_info[relative_path] = local_path.stat().st_size
    return file_info


repo_id = "mimbres/YourMT3"
yourmt3_dir = "/content/music-to-midi/space/YourMT3"
amt_src = os.path.join(yourmt3_dir, "amt", "src")

log("准备 YourMT3 源码同步（详细模式）")
log(f"目标目录: {yourmt3_dir}")
log(
    "HF_TOKEN 状态: 已设置"
    if os.environ.get("HF_TOKEN")
    else "HF_TOKEN 状态: 未设置（公开仓库可匿名下载）"
)

source_start = time.time()
repo_source_files = sorted(
    file_path
    for file_path in list_repo_files(repo_id, repo_type="space")
    if file_path.startswith("amt/src/")
)

if not repo_source_files:
    raise RuntimeError("远程仓库中未找到 amt/src 文件，请重试。")

log(f"远程源文件数量: {len(repo_source_files)}")
source_before = collect_local_file_info(yourmt3_dir, repo_source_files)
log(f"同步前本地缓存文件数: {len(source_before)}")
log("源文件计划（同步前）：")
for index, file_path in enumerate(repo_source_files, 1):
    if file_path in source_before:
        log(f"  [{index:03d}/{len(repo_source_files)}] CACHE {file_path} ({format_size(source_before[file_path])})")
    else:
        log(f"  [{index:03d}/{len(repo_source_files)}] FETCH {file_path}")

snapshot_download(
    repo_id,
    repo_type="space",
    local_dir=yourmt3_dir,
    allow_patterns=["amt/src/**"],
    ignore_patterns=["*.ckpt", "*.bin", "*.safetensors", "amt/logs/**"],
)

source_after = collect_local_file_info(yourmt3_dir, repo_source_files)
new_files = [file_path for file_path in repo_source_files if file_path not in source_before and file_path in source_after]
cached_files = [file_path for file_path in repo_source_files if file_path in source_before and file_path in source_after]

log("源文件结果（同步后）：")
for index, file_path in enumerate(repo_source_files, 1):
    if file_path in source_after:
        status = "NEW" if file_path in new_files else "CACHE"
        log(f"  [{index:03d}/{len(repo_source_files)}] {status:5} {file_path} ({format_size(source_after[file_path])})")
    else:
        log(f"  [{index:03d}/{len(repo_source_files)}] MISSING {file_path}")

source_total_size = sum(source_after.values())
source_elapsed = time.time() - source_start
log(
    f"YourMT3 源码就绪: 新增={len(new_files)}, 缓存={len(cached_files)}, "
    f"总计={format_size(source_total_size)}, 耗时={source_elapsed:.1f}秒"
)

root_link = "/content/music-to-midi/YourMT3"
if not os.path.exists(root_link):
    os.symlink(yourmt3_dir, root_link)
    log("符号链接已创建: YourMT3 -> space/YourMT3")
else:
    log("符号链接已存在: YourMT3 -> space/YourMT3")

if amt_src not in sys.path:
    sys.path.insert(0, amt_src)
    log("已将 amt/src 添加到 sys.path")

log("开始检查/下载 YourMT3+ 官方模型模式权重（首次运行会下载全部可选模式）")
sys.path.insert(0, "/content/music-to-midi")
from download_sota_models import download_official_yourmt3_models

model_cache_root = Path(os.path.expanduser("~/.cache/music_ai_models/yourmt3_all"))
existing_ckpts = sorted(model_cache_root.rglob("*.ckpt")) if model_cache_root.exists() else []
if existing_ckpts:
    log("现有检查点缓存：")
    for idx, ckpt in enumerate(existing_ckpts, 1):
        log(f"  [{idx:02d}/{len(existing_ckpts)}] {ckpt} ({format_size(ckpt.stat().st_size)})")
else:
    log("本地未找到检查点缓存；预计首次下载")

downloaded_yourmt3_models = download_official_yourmt3_models()
log(f"YourMT3+ 官方模型模式准备完成: {len(downloaded_yourmt3_models)} 个")

final_ckpts = sorted(model_cache_root.rglob("*.ckpt")) if model_cache_root.exists() else []
if final_ckpts:
    log("下载后的检查点文件：")
    for idx, ckpt in enumerate(final_ckpts, 1):
        log(f"  [{idx:02d}/{len(final_ckpts)}] {ckpt} ({format_size(ckpt.stat().st_size)})")
else:
    log("警告：下载后未检测到 .ckpt 文件，请检查上方日志。")

log("检查六声部分离、Aria-AMT 与 ByteDance Piano 资源（按需下载）")
from download_multistem_model import download_multistem_model
from download_aria_amt_model import download_aria_model
from download_bytedance_piano_model import download_bytedance_piano_model

log("准备六声部分离资源: download_multistem_model.py")
download_multistem_model(printer=log)
log("准备 Aria-AMT 资源: download_aria_amt_model.py")
download_aria_model(printer=log)
log("准备 ByteDance Piano 资源: download_bytedance_piano_model.py")
download_bytedance_piano_model(printer=log)

log("所有模型就绪")

In [ ]:
#@title 4️⃣ 启动 Gradio 应用（详细日志 + 公开链接）
import logging
import os
import platform
import subprocess
import sys
import tempfile
from datetime import datetime
from pathlib import Path


def tlog(message=""):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


os.chdir("/content/music-to-midi")
try:
    subprocess.run(["fuser", "-k", "7860/tcp"], capture_output=True, text=True, timeout=5)
except Exception as exc:
    tlog(f"Skip port cleanup: {exc}")

PROJECT_ROOT = "/content/music-to-midi"
amt_src = "/content/music-to-midi/space/YourMT3/amt/src"
for p in [PROJECT_ROOT, amt_src]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_MIN_LOG_LEVEL"] = "3"
os.environ["NUMBA_CACHE_DIR"] = "/tmp/numba_cache"
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
logger = logging.getLogger("colab-app")

import gradio as gr
import torch

logger.info("========== Startup Diagnostics ==========")
logger.info(f"Python version: {platform.python_version()}")
logger.info(f"Gradio version: {gr.__version__}")
logger.info(f"Torch version: {torch.__version__}")
logger.info(f"CUDA available: {torch.cuda.is_available()}")
logger.info(f"Project root: {PROJECT_ROOT}")
logger.info(f"YourMT3 source path: {amt_src}")
logger.info(f"YourMT3 ymt3.py exists: {Path(amt_src, 'model', 'ymt3.py').exists()}")

try:
    from model.ymt3 import YourMT3  # noqa: F401
    logger.info("✅ YourMT3 code import success")
except ImportError as exc:
    raise RuntimeError(f"YourMT3 import failed: {exc}, 请重新运行第 3 步")

from src.models.data_models import Config, ProcessingStage
from src.core.pipeline import MusicToMidiPipeline
from src.utils.gpu_utils import clear_gpu_memory
from src.utils.yourmt3_downloader import DEFAULT_MODEL, OFFICIAL_YOURMT3_MODEL_KEYS, YOURMT3_MODELS

device_label = f"GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU"
logger.info(f"Current device: {device_label}")

model_cache_root = Path(os.path.expanduser("~/.cache/music_ai_models/yourmt3_all"))
ckpts = sorted(model_cache_root.rglob("*.ckpt")) if model_cache_root.exists() else []
logger.info(f"Model cache root: {model_cache_root}")
logger.info(f"Checkpoint count: {len(ckpts)}")

YOURMT3_MODEL_LABEL_TO_KEY = {
    YOURMT3_MODELS[model_key].get("ui_label", model_key): model_key
    for model_key in OFFICIAL_YOURMT3_MODEL_KEYS
}
YOURMT3_MODEL_CHOICES = list(YOURMT3_MODEL_LABEL_TO_KEY.keys())
DEFAULT_YOURMT3_MODEL_LABEL = next(
    label for label, model_key in YOURMT3_MODEL_LABEL_TO_KEY.items() if model_key == DEFAULT_MODEL
)
YOURMT3_PROCESSING_MODES = {"smart", "vocal_split", "six_stem_split"}


def uses_yourmt3_processing_mode(mode):
    return mode in {
        "完整混音多乐器转写（SMART）",
        "人声/伴奏分离后分别转写（VOCAL_SPLIT）",
        "六声部分离 + 分别转写",
    }


LOG_FILE = "/tmp/midi_process.log"


class _RobustFileHandler(logging.Handler):
    def __init__(self, filename):
        super().__init__()
        self.filename = filename

    def emit(self, record):
        try:
            msg = self.format(record)
            with open(self.filename, "a", encoding="utf-8") as f:
                f.write(msg + "\n")
        except Exception:
            pass


_fh = _RobustFileHandler(LOG_FILE)
_fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S"))
_fh.setLevel(logging.INFO)
for _name in ("colab-app", "src.core", "src.utils"):
    logging.getLogger(_name).addHandler(_fh)


def clear_logs():
    try:
        open(LOG_FILE, "w", encoding="utf-8").close()
    except Exception:
        pass


def read_logs():
    try:
        with open(LOG_FILE, "r", encoding="utf-8", errors="replace") as f:
            content = f.read().replace("\x00", "")
        lines = content.strip().split("\n")
        return "\n".join(lines[-200:]) if lines and lines[0] else ""
    except Exception as exc:
        return f"[error] {exc}"


def ensure_aria_amt_weights():
    from download_aria_amt_model import download_aria_model, is_aria_model_available

    if is_aria_model_available():
        logger.info("Aria-AMT checkpoint found")
        return
    logger.info("Aria-AMT checkpoint not found, downloading via download_aria_amt_model.py")
    download_aria_model(printer=logger.info)


def convert_audio_to_midi(
    audio_path,
    mode,
    yourmt3_model_label,
    vocal_split_merge_midi=False,
    six_stem_only_selected=False,
    six_stem_targets=None,
    six_stem_vocal_harmony=False,
    progress=gr.Progress(),
):
    if audio_path is None:
        raise gr.Error("请先上传音频文件")

    clear_logs()
    logger.info("========== New Conversion ==========")
    logger.info(f"Audio file: {Path(audio_path).name}")
    logger.info(f"Mode: {mode}")

    config = Config()
    mode_mapping = {
        "完整混音多乐器转写（SMART）": "smart",
        "人声/伴奏分离后分别转写（VOCAL_SPLIT）": "vocal_split",
        "六声部分离 + 分别转写": "six_stem_split",
        "钢琴专用转写 (Transkun)": "piano_transkun",
        "钢琴专用转写 (Aria-AMT)": "piano_aria_amt",
        "钢琴专用转写 (ByteDance Pedal)": "piano_bytedance_pedal",
    }
    config.processing_mode = mode_mapping.get(mode, "smart")
    yourmt3_model_key = YOURMT3_MODEL_LABEL_TO_KEY.get(yourmt3_model_label)
    if yourmt3_model_key is None:
        raise gr.Error(f"未知 YourMT3+ 模型模式: {yourmt3_model_label}")
    if config.processing_mode in YOURMT3_PROCESSING_MODES:
        config.yourmt3_model = yourmt3_model_key
        logger.info(f"YourMT3 model mode: {yourmt3_model_label} ({yourmt3_model_key})")
    config.vocal_split_merge_midi = bool(config.processing_mode == "vocal_split" and vocal_split_merge_midi)
    if config.processing_mode == "six_stem_split" and six_stem_only_selected:
        config.six_stem_targets = [str(stem).lower() for stem in (six_stem_targets or [])]
    else:
        config.six_stem_targets = []
    config.six_stem_split_vocal_harmony = bool(config.processing_mode == "six_stem_split" and six_stem_vocal_harmony)

    if config.processing_mode == "piano_aria_amt":
        ensure_aria_amt_weights()
    elif config.processing_mode == "piano_bytedance_pedal":
        from download_bytedance_piano_model import download_bytedance_piano_model
        download_bytedance_piano_model(printer=logger.info)

    pipeline = MusicToMidiPipeline(config)
    output_dir = tempfile.mkdtemp(prefix="midi_output_")
    logger.info(f"Output dir: {output_dir}")

    def on_progress(p):
        stage_name = {
            ProcessingStage.PREPROCESSING: "预处理",
            ProcessingStage.SEPARATION: "音源分离",
            ProcessingStage.TRANSCRIPTION: "音频转写",
            ProcessingStage.VOCAL_TRANSCRIPTION: "人声转写",
            ProcessingStage.SYNTHESIS: "MIDI合成",
            ProcessingStage.COMPLETE: "完成",
        }.get(p.stage, str(p.stage))
        logger.info(f"Progress {p.overall_progress * 100:5.1f}% [{stage_name}] {p.message}")
        progress(p.overall_progress, desc=f"[{stage_name}] {p.message}")

    try:
        result = pipeline.process(audio_path=audio_path, output_dir=output_dir, progress_callback=on_progress)
    except Exception as exc:
        logger.error(f"Conversion failed: {exc}")
        raise gr.Error(f"转换失败: {exc}") from exc
    finally:
        try:
            clear_gpu_memory()
            logger.info("GPU memory cleaned")
        except Exception as exc:
            logger.warning(f"GPU memory cleanup warning: {exc}")

    output_files = []
    if result.midi_path and Path(result.midi_path).exists():
        output_files.append(result.midi_path)
    if result.stem_midi_paths:
        for stem_midi_path in result.stem_midi_paths.values():
            if stem_midi_path and Path(stem_midi_path).exists():
                output_files.append(stem_midi_path)
    if result.vocal_midi_path and Path(result.vocal_midi_path).exists():
        output_files.append(result.vocal_midi_path)
    if result.accompaniment_midi_path and Path(result.accompaniment_midi_path).exists():
        if result.accompaniment_midi_path != result.midi_path:
            output_files.append(result.accompaniment_midi_path)
    if result.separated_audio:
        for audio_file in result.separated_audio.values():
            if audio_file and Path(audio_file).exists():
                output_files.append(audio_file)

    output_files = list(dict.fromkeys(output_files))
    logger.info(f"Output file count: {len(output_files)}")

    bpm_str = f"{result.beat_info.bpm:.1f}" if result.beat_info else "N/A"
    status_lines = [
        "--- 转换完成 ---",
        f"耗时: {result.processing_time:.1f} 秒",
        f"总音符数: {result.total_notes}",
        f"BPM: {bpm_str}",
        f"设备: {device_label}",
        f"输出文件数: {len(output_files)}",
    ]
    if result.stem_midi_paths:
        status_lines.append(f"合并 MIDI: {Path(result.midi_path).name}")
        status_lines.append(f"分 stem MIDI: {len(result.stem_midi_paths)} 个")
    elif result.vocal_midi_path:
        status_lines.append(f"伴奏 MIDI: {Path(result.accompaniment_midi_path).name}")
        status_lines.append(f"人声 MIDI: {Path(result.vocal_midi_path).name}")
        if result.merged_midi_path:
            status_lines.append(f"合并 MIDI: {Path(result.merged_midi_path).name}")
    else:
        status_lines.append(f"MIDI 文件: {Path(result.midi_path).name}")

    logger.info("========== Conversion Finished ==========")
    return output_files, "\n".join(status_lines)


def update_mode_info(mode):
    if mode == "人声/伴奏分离后分别转写（VOCAL_SPLIT）":
        return "**VOCAL_SPLIT** — 使用 BS-RoFormer 分离人声与伴奏，再用 YourMT3+ 分别转写。"
    if mode == "六声部分离 + 分别转写":
        return "**六声部分离 + 分别转写** — 分离 bass/drums/guitar/piano/vocals/other 六个 stem，再分别转写并合并 MIDI。"
    if mode == "钢琴专用转写 (Transkun)":
        return "**Transkun** — 钢琴专用转写，适合纯钢琴音频。"
    if mode == "钢琴专用转写 (Aria-AMT)":
        return "**Aria-AMT** — 钢琴专用转写，首次使用会检查 Aria-AMT checkpoint。"
    if mode == "钢琴专用转写 (ByteDance Pedal)":
        return "**ByteDance Pedal** — 钢琴专用带踏板转写，适合纯钢琴音频，会保留延音踏板 CC64。"
    return "**SMART** — 通过所选 YourMT3+ 官方模型模式对完整混音进行多乐器转写，精确识别 128 种 GM 乐器。"


def update_yourmt3_model_info(yourmt3_model_label):
    model_key = YOURMT3_MODEL_LABEL_TO_KEY.get(yourmt3_model_label, DEFAULT_MODEL)
    model_info = YOURMT3_MODELS[model_key]
    label = model_info.get("ui_label", yourmt3_model_label)
    description = model_info.get("description") or model_info.get("ui_description", "")
    features = ", ".join(model_info.get("features", []))
    checkpoint = model_info.get("checkpoint", "")
    return (
        f"**YourMT3+ 模型模式：{label}**\n\n"
        f"{description}\n\n"
        f"**Checkpoint**: `{checkpoint}`\n\n"
        f"**Features**: {features}"
    )


def update_output_info(mode):
    if mode == "人声/伴奏分离后分别转写（VOCAL_SPLIT）":
        return "**输出说明**\n\n- 默认输出 `_accompaniment.mid` 与 `_vocal.mid`。\n- 勾选合并后额外输出 `_vocal_accompaniment_merged.mid`。"
    if mode == "六声部分离 + 分别转写":
        return "**输出说明**\n\n- 输出分离后的 stem 音频。\n- 输出选中 stem 的独立 MIDI。\n- 额外输出一个合并 MIDI。"
    return "**输出说明**\n\n- 输出一个 MIDI 文件。"


def update_mode_controls(mode, only_selected):
    is_vocal = mode == "人声/伴奏分离后分别转写（VOCAL_SPLIT）"
    is_six = mode == "六声部分离 + 分别转写"
    uses_yourmt3 = uses_yourmt3_processing_mode(mode)
    return (
        update_mode_info(mode),
        update_output_info(mode),
        gr.update(visible=uses_yourmt3),
        gr.update(visible=uses_yourmt3),
        gr.update(visible=is_vocal),
        gr.update(visible=is_six),
        gr.update(visible=(is_six and bool(only_selected))),
        gr.update(visible=is_six),
    )


def update_six_stem_targets_visibility(mode, only_selected):
    return gr.update(visible=(mode == "六声部分离 + 分别转写" and bool(only_selected)))


LOG_POLL_JS = """<script>
(function() {
    var _pollTimer = setInterval(function() {
        var ta = document.querySelector('.log-box textarea');
        if (!ta) return;
        var setter = Object.getOwnPropertyDescriptor(HTMLTextAreaElement.prototype, 'value').set;
        fetch('./api/read_logs', {method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify({data: []})})
        .then(function(r) { return r.json(); })
        .then(function(json) {
            var logText = (json.data && json.data[0]) ? json.data[0] : '';
            if (logText) setter.call(ta, logText);
            ta.dispatchEvent(new Event('input', {bubbles: true}));
            ta.scrollTop = ta.scrollHeight;
        })
        .catch(function() {});
    }, 1200);
})();
</script>"""

with gr.Blocks(
    title="Music to MIDI (Colab GPU)",
    head=LOG_POLL_JS,
    theme=gr.themes.Base(primary_hue=gr.themes.colors.blue, neutral_hue=gr.themes.colors.slate),
) as demo:
    gr.Markdown("# 🎵 Music to MIDI\n当前 Colab 版提供六种处理模式，SMART/VOCAL_SPLIT/六声部分离使用可选择的 YourMT3+ 官方模型模式，钢琴专用模式使用 Transkun、Aria-AMT 或 ByteDance Pedal。")

    with gr.Row():
        with gr.Column(scale=5):
            audio_input = gr.Audio(label="上传音频文件", type="filepath")
            gr.Markdown("<small>支持 MP3, WAV, FLAC, OGG, M4A（自动转换为 WAV 处理）</small>")
            mode_radio = gr.Radio(
                [
                    "完整混音多乐器转写（SMART）",
                    "人声/伴奏分离后分别转写（VOCAL_SPLIT）",
                    "六声部分离 + 分别转写",
                    "钢琴专用转写 (Transkun)",
                    "钢琴专用转写 (Aria-AMT)",
                    "钢琴专用转写 (ByteDance Pedal)",
                ],
                value="完整混音多乐器转写（SMART）",
                label="处理模式",
            )
            mode_info = gr.Markdown(update_mode_info("完整混音多乐器转写（SMART）"))
            yourmt3_model_dropdown = gr.Dropdown(
                choices=YOURMT3_MODEL_CHOICES,
                value=DEFAULT_YOURMT3_MODEL_LABEL,
                label="YourMT3+ 官方模型模式",
                info="SMART / VOCAL_SPLIT / 六声部分离模式使用该模型设置",
            )
            yourmt3_model_info = gr.Markdown(update_yourmt3_model_info(DEFAULT_YOURMT3_MODEL_LABEL))
            output_info = gr.Markdown(update_output_info("完整混音多乐器转写（SMART）"))
            vocal_split_merge_midi = gr.Checkbox(value=False, label="输出 1 个人声+伴奏合并 MIDI（人声分离模式）", visible=False)
            six_stem_only_selected = gr.Checkbox(value=False, label="仅转写选中的 stem（六声部模式）", visible=False)
            six_stem_targets = gr.CheckboxGroup(
                choices=["bass", "drums", "guitar", "piano", "vocals", "other"],
                value=["bass", "drums", "guitar", "piano", "vocals", "other"],
                label="六声部转写目标",
                info="仅在勾选“仅转写选中的 stem”且模式为六声部分离时生效",
                visible=False,
            )
            six_stem_vocal_harmony = gr.Checkbox(value=False, label="将 vocals 进一步分离为主唱 + 和声（实验近似）", visible=False)
            mode_radio.change(
                fn=update_mode_controls,
                inputs=[mode_radio, six_stem_only_selected],
                outputs=[mode_info, output_info, yourmt3_model_dropdown, yourmt3_model_info, vocal_split_merge_midi, six_stem_only_selected, six_stem_targets, six_stem_vocal_harmony],
                api_name=False,
            )
            yourmt3_model_dropdown.change(
                fn=update_yourmt3_model_info,
                inputs=[yourmt3_model_dropdown],
                outputs=[yourmt3_model_info],
                api_name=False,
            )
            six_stem_only_selected.change(
                fn=update_six_stem_targets_visibility,
                inputs=[mode_radio, six_stem_only_selected],
                outputs=[six_stem_targets],
                api_name=False,
            )
            convert_btn = gr.Button("▶ 开始转换", variant="primary", size="lg")
            gr.Markdown(f"当前设备: **{device_label}**")

        with gr.Column(scale=5):
            status_output = gr.Textbox(label="状态", interactive=False, lines=7, placeholder="等待转换...")
            file_output = gr.File(label="下载文件", file_count="multiple")
            log_output = gr.Textbox(label="控制台日志", interactive=False, lines=16, max_lines=30, elem_classes="log-box")

    convert_btn.click(
        fn=convert_audio_to_midi,
        inputs=[audio_input, mode_radio, yourmt3_model_dropdown, vocal_split_merge_midi, six_stem_only_selected, six_stem_targets, six_stem_vocal_harmony],
        outputs=[file_output, status_output],
    )

    _log_poll_btn = gr.Button(visible=False)
    _log_poll_btn.click(fn=read_logs, inputs=[], outputs=[log_output], api_name="read_logs", queue=False)

    gr.Markdown("<center>基于 [YourMT3+](https://github.com/mimbres/YourMT3) 官方模型模式 | [GitHub](https://github.com/mason369/music-to-midi)</center>")

print("\n" + "=" * 60)
print("🚀 启动中... 公开链接将在下方显示（详细日志模式）")
print("=" * 60 + "\n")
demo.launch(share=True, server_name="0.0.0.0", server_port=7860)
